<a href="https://colab.research.google.com/github/GautamAjesh/A-Comparative-Analysis-of-Quantization-Methods-Across-NLP-Tasks-in-Small-Language-Models/blob/main/quantization_nlp_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torch -q

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

print("Model loaded successfully!")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded successfully!


In [3]:
import torch

text = "This movie was absolutely wonderful!"

inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

print("Raw output (logits):", outputs.logits)

predicted_class = torch.argmax(outputs.logits, dim=1).item()
labels = model.config.id2label
print("Predicted sentiment:", labels[predicted_class])

Raw output (logits): tensor([[-4.3413,  4.6803]])
Predicted sentiment: POSITIVE


In [4]:
import os

def get_model_size(model, label="model"):
    torch.save(model.state_dict(), "temp.pt")
    size_mb = os.path.getsize("temp.pt") / (1024 * 1024)
    os.remove("temp.pt")
    print(f"{label} size: {size_mb:.2f} MB")
    return size_mb

fp32_size = get_model_size(model, "FP32 baseline")

FP32 baseline size: 255.45 MB


In [5]:
import torch.quantization

quantized_model = torch.quantization.quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)

print("Quantized model created!")

/tmp/ipykernel_502/7625546.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


Quantized model created!


In [6]:
int8_size = get_model_size(quantized_model, "INT8 Dynamic Quantized")

print(f"\nSize reduction: {fp32_size - int8_size:.2f} MB")
print(f"Percentage reduction: {((fp32_size - int8_size) / fp32_size) * 100:.2f}%")

INT8 Dynamic Quantized size: 132.28 MB

Size reduction: 123.16 MB
Percentage reduction: 48.22%


In [7]:
with torch.no_grad():
    outputs_quant = quantized_model(**inputs)

predicted_class_quant = torch.argmax(outputs_quant.logits, dim=1).item()
print("Quantized model prediction:", labels[predicted_class_quant])
print("Quantized logits:", outputs_quant.logits)

print("\nOriginal logits:   ", outputs.logits)
print("Quantized logits:  ", outputs_quant.logits)

Quantized model prediction: POSITIVE
Quantized logits: tensor([[-4.3294,  4.6849]])

Original logits:    tensor([[-4.3413,  4.6803]])
Quantized logits:   tensor([[-4.3294,  4.6849]])


In [8]:
!pip install datasets -q

from datasets import load_dataset

dataset = load_dataset("stanfordnlp/sst2", split="validation[:200]")
print(dataset)
print(dataset[0])

README.md:   0%|          | 0.00/5.27k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Dataset({
    features: ['idx', 'sentence', 'label'],
    num_rows: 200
})
{'idx': 0, 'sentence': "it 's a charming and often affecting journey . ", 'label': 1}


In [9]:
def evaluate_model(model, dataset, tokenizer):
    correct = 0
    total = len(dataset)

    for example in dataset:
        text = example['sentence']
        true_label = example['label']

        inputs = tokenizer(text, return_tensors="pt")
        with torch.no_grad():
            outputs = model(**inputs)

        predicted_label = torch.argmax(outputs.logits, dim=1).item()

        if predicted_label == true_label:
            correct += 1

    accuracy = correct / total
    return accuracy

fp32_accuracy = evaluate_model(model, dataset, tokenizer)
print(f"FP32 Accuracy: {fp32_accuracy * 100:.2f}%")

FP32 Accuracy: 91.00%


In [10]:
int8_accuracy = evaluate_model(quantized_model, dataset, tokenizer)
print(f"INT8 Accuracy: {int8_accuracy * 100:.2f}%")

print(f"\nAccuracy drop: {(fp32_accuracy - int8_accuracy) * 100:.2f} percentage points")

INT8 Accuracy: 91.00%

Accuracy drop: 0.00 percentage points


In [11]:
model_name_nli = "textattack/distilbert-base-uncased-RTE"

tokenizer_nli = AutoTokenizer.from_pretrained(model_name_nli)
model_nli = AutoModelForSequenceClassification.from_pretrained(model_name_nli)

print("NLI model loaded!")
print(model_nli.config.id2label)

config.json:   0%|          | 0.00/489 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  268MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

NLI model loaded!
{0: 'LABEL_0', 1: 'LABEL_1'}


In [12]:
dataset_rte = load_dataset("nyu-mll/glue", "rte", split="validation[:200]")
print(dataset_rte)
print(dataset_rte[0])

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

rte/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  584kB            

rte/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

rte/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2490 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/277 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 200
})
{'sentence1': 'Dana Reeve, the widow of the actor Christopher Reeve, has died of lung cancer at age 44, according to the Christopher Reeve Foundation.', 'sentence2': 'Christopher Reeve had an accident.', 'label': 1, 'idx': 0}


In [13]:
def evaluate_nli_model(model, dataset, tokenizer):
    correct = 0
    total = len(dataset)

    for example in dataset:
        premise = example['sentence1']
        hypothesis = example['sentence2']
        true_label = example['label']

        inputs = tokenizer(premise, hypothesis, return_tensors="pt", truncation=True)
        with torch.no_grad():
            outputs = model(**inputs)

        predicted_label = torch.argmax(outputs.logits, dim=1).item()

        if predicted_label == true_label:
            correct += 1

    accuracy = correct / total
    return accuracy

fp32_accuracy_rte = evaluate_nli_model(model_nli, dataset_rte, tokenizer_nli)
print(f"FP32 NLI (RTE) Accuracy: {fp32_accuracy_rte * 100:.2f}%")

FP32 NLI (RTE) Accuracy: 67.50%


In [14]:
quantized_model_nli = torch.quantization.quantize_dynamic(
    model_nli, {torch.nn.Linear}, dtype=torch.qint8
)

int8_accuracy_rte = evaluate_nli_model(quantized_model_nli, dataset_rte, tokenizer_nli)
print(f"INT8 NLI (RTE) Accuracy: {int8_accuracy_rte * 100:.2f}%")

print(f"\nAccuracy drop: {(fp32_accuracy_rte - int8_accuracy_rte) * 100:.2f} percentage points")

/tmp/ipykernel_502/2447083868.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model_nli = torch.quantization.quantize_dynamic(


INT8 NLI (RTE) Accuracy: 61.50%

Accuracy drop: 6.00 percentage points


In [15]:
from transformers import AutoModelForQuestionAnswering

model_name_qa = "distilbert-base-uncased-distilled-squad"

tokenizer_qa = AutoTokenizer.from_pretrained(model_name_qa)
model_qa = AutoModelForQuestionAnswering.from_pretrained(model_name_qa)

print("QA model loaded!")

config.json:   0%|          | 0.00/451 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  265MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

QA model loaded!


In [16]:
dataset_qa = load_dataset("rajpurkar/squad", split="validation[:200]")
print(dataset_qa)
print(dataset_qa[0])

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 200
})
{'id': '56be4db0acb8001400a502ec', 'title': 'Super_Bowl_50', 'context': 'Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.', 'question': 'Which NFL team represented the AFC at S

In [17]:
def get_answer(model, tokenizer, question, context):
    inputs = tokenizer(question, context, return_tensors="pt", truncation=True, max_length=384)
    with torch.no_grad():
        outputs = model(**inputs)

    start_idx = torch.argmax(outputs.start_logits)
    end_idx = torch.argmax(outputs.end_logits)

    answer_tokens = inputs["input_ids"][0][start_idx:end_idx+1]
    answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)
    return answer

# Quick test on the first example
example = dataset_qa[0]
predicted = get_answer(model_qa, tokenizer_qa, example['question'], example['context'])
print("Question:", example['question'])
print("Predicted answer:", predicted)
print("Correct answer(s):", example['answers']['text'])

Question: Which NFL team represented the AFC at Super Bowl 50?
Predicted answer: denver broncos
Correct answer(s): ['Denver Broncos', 'Denver Broncos', 'Denver Broncos']


In [18]:
import re, string
from collections import Counter

def normalize_text(s):
    s = s.lower()
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ' '.join(s.split())
    return s

def compute_f1(prediction, truth):
    pred_tokens = normalize_text(prediction).split()
    truth_tokens = normalize_text(truth).split()

    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return int(pred_tokens == truth_tokens)

    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1

def evaluate_qa_model(model, tokenizer, dataset):
    f1_scores = []

    for example in dataset:
        predicted = get_answer(model, tokenizer, example['question'], example['context'])
        best_f1 = max(compute_f1(predicted, ans) for ans in example['answers']['text'])
        f1_scores.append(best_f1)

    avg_f1 = sum(f1_scores) / len(f1_scores)
    return avg_f1

fp32_f1_qa = evaluate_qa_model(model_qa, tokenizer_qa, dataset_qa)
print(f"FP32 QA (SQuAD) F1 Score: {fp32_f1_qa * 100:.2f}")

FP32 QA (SQuAD) F1 Score: 82.11


In [19]:
quantized_model_qa = torch.quantization.quantize_dynamic(
    model_qa, {torch.nn.Linear}, dtype=torch.qint8
)

int8_f1_qa = evaluate_qa_model(quantized_model_qa, tokenizer_qa, dataset_qa)
print(f"INT8 QA (SQuAD) F1 Score: {int8_f1_qa * 100:.2f}")

print(f"\nF1 drop: {(fp32_f1_qa - int8_f1_qa) * 100:.2f} points")

/tmp/ipykernel_502/791405859.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model_qa = torch.quantization.quantize_dynamic(


INT8 QA (SQuAD) F1 Score: 80.13

F1 drop: 1.98 points


In [20]:
import time

def measure_latency(predict_fn, n_runs=50):
    # Warm-up run (first run is always slower due to caching)
    predict_fn()

    start = time.time()
    for _ in range(n_runs):
        predict_fn()
    end = time.time()

    avg_latency_ms = ((end - start) / n_runs) * 1000
    return avg_latency_ms

# Sentiment latency
sentiment_text = dataset[0]['sentence']
sentiment_inputs = tokenizer(sentiment_text, return_tensors="pt")

fp32_latency_sent = measure_latency(lambda: model(**sentiment_inputs))
int8_latency_sent = measure_latency(lambda: quantized_model(**sentiment_inputs))

print(f"Sentiment - FP32 latency: {fp32_latency_sent:.2f} ms | INT8 latency: {int8_latency_sent:.2f} ms")

Sentiment - FP32 latency: 56.73 ms | INT8 latency: 14.47 ms


In [21]:
# NLI latency
nli_example = dataset_rte[0]
nli_inputs = tokenizer_nli(nli_example['sentence1'], nli_example['sentence2'], return_tensors="pt", truncation=True)

fp32_latency_nli = measure_latency(lambda: model_nli(**nli_inputs))
int8_latency_nli = measure_latency(lambda: quantized_model_nli(**nli_inputs))

print(f"NLI - FP32 latency: {fp32_latency_nli:.2f} ms | INT8 latency: {int8_latency_nli:.2f} ms")

# QA latency
qa_example = dataset_qa[0]
qa_inputs = tokenizer_qa(qa_example['question'], qa_example['context'], return_tensors="pt", truncation=True, max_length=384)

fp32_latency_qa = measure_latency(lambda: model_qa(**qa_inputs))
int8_latency_qa = measure_latency(lambda: quantized_model_qa(**qa_inputs))

print(f"QA - FP32 latency: {fp32_latency_qa:.2f} ms | INT8 latency: {int8_latency_qa:.2f} ms")

NLI - FP32 latency: 68.28 ms | INT8 latency: 30.21 ms
QA - FP32 latency: 214.51 ms | INT8 latency: 132.70 ms


In [22]:
fp32_size_nli = get_model_size(model_nli, "NLI FP32")
int8_size_nli = get_model_size(quantized_model_nli, "NLI INT8")

fp32_size_qa = get_model_size(model_qa, "QA FP32")
int8_size_qa = get_model_size(quantized_model_qa, "QA INT8")

NLI FP32 size: 255.45 MB
NLI INT8 size: 132.28 MB
QA FP32 size: 253.19 MB
QA INT8 size: 131.72 MB


In [23]:
dataset_large = load_dataset("stanfordnlp/sst2", split="validation[:500]")
dataset_rte_large = load_dataset("nyu-mll/glue", "rte", split="validation")  # full set, 277 examples
dataset_qa_large = load_dataset("rajpurkar/squad", split="validation[:500]")

print(f"Sentiment: {len(dataset_large)} examples")
print(f"RTE: {len(dataset_rte_large)} examples")
print(f"QA: {len(dataset_qa_large)} examples")

Sentiment: 500 examples
RTE: 277 examples
QA: 500 examples


In [24]:
fp32_accuracy_large = evaluate_model(model, dataset_large, tokenizer)
int8_accuracy_large = evaluate_model(quantized_model, dataset_large, tokenizer)

print(f"FP32 Sentiment Accuracy (n=500): {fp32_accuracy_large * 100:.2f}%")
print(f"INT8 Sentiment Accuracy (n=500): {int8_accuracy_large * 100:.2f}%")
print(f"Drop: {(fp32_accuracy_large - int8_accuracy_large) * 100:.2f} pts")

FP32 Sentiment Accuracy (n=500): 91.20%
INT8 Sentiment Accuracy (n=500): 90.60%
Drop: 0.60 pts


In [25]:
fp32_accuracy_rte_large = evaluate_nli_model(model_nli, dataset_rte_large, tokenizer_nli)
int8_accuracy_rte_large = evaluate_nli_model(quantized_model_nli, dataset_rte_large, tokenizer_nli)

print(f"FP32 NLI Accuracy (n=277): {fp32_accuracy_rte_large * 100:.2f}%")
print(f"INT8 NLI Accuracy (n=277): {int8_accuracy_rte_large * 100:.2f}%")
print(f"Drop: {(fp32_accuracy_rte_large - int8_accuracy_rte_large) * 100:.2f} pts")

FP32 NLI Accuracy (n=277): 64.98%
INT8 NLI Accuracy (n=277): 60.65%
Drop: 4.33 pts


In [26]:
fp32_f1_qa_large = evaluate_qa_model(model_qa, tokenizer_qa, dataset_qa_large)
int8_f1_qa_large = evaluate_qa_model(quantized_model_qa, tokenizer_qa, dataset_qa_large)

print(f"FP32 QA F1 (n=500): {fp32_f1_qa_large * 100:.2f}")
print(f"INT8 QA F1 (n=500): {int8_f1_qa_large * 100:.2f}")
print(f"Drop: {(fp32_f1_qa_large - int8_f1_qa_large) * 100:.2f} pts")

FP32 QA F1 (n=500): 81.92
INT8 QA F1 (n=500): 76.26
Drop: 5.66 pts


In [27]:
!pip install statsmodels -q


In [28]:
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np

def get_predictions(model, dataset, tokenizer, task="sentiment"):
    preds = []
    for example in dataset:
        if task == "sentiment":
            inputs = tokenizer(example['sentence'], return_tensors="pt")
        elif task == "nli":
            inputs = tokenizer(example['sentence1'], example['sentence2'], return_tensors="pt", truncation=True)
        with torch.no_grad():
            outputs = model(**inputs)
        preds.append(torch.argmax(outputs.logits, dim=1).item())
    return preds

def mcnemar_test(fp32_preds, int8_preds, true_labels):
    fp32_correct = np.array(fp32_preds) == np.array(true_labels)
    int8_correct = np.array(int8_preds) == np.array(true_labels)

    # Contingency table: [both correct, fp32 right int8 wrong], [fp32 wrong int8 right, both wrong]
    both_correct = np.sum(fp32_correct & int8_correct)
    fp32_only = np.sum(fp32_correct & ~int8_correct)
    int8_only = np.sum(~fp32_correct & int8_correct)
    both_wrong = np.sum(~fp32_correct & ~int8_correct)

    table = [[both_correct, fp32_only], [int8_only, both_wrong]]
    result = mcnemar(table, exact=True if (fp32_only + int8_only) < 25 else False)
    return result, table

# Sentiment
sent_true = [ex['label'] for ex in dataset_large]
sent_fp32_preds = get_predictions(model, dataset_large, tokenizer, "sentiment")
sent_int8_preds = get_predictions(quantized_model, dataset_large, tokenizer, "sentiment")
result_sent, table_sent = mcnemar_test(sent_fp32_preds, sent_int8_preds, sent_true)
print("Sentiment McNemar's test:", result_sent)
print("Contingency table:", table_sent)

# NLI
nli_true = [ex['label'] for ex in dataset_rte_large]
nli_fp32_preds = get_predictions(model_nli, dataset_rte_large, tokenizer_nli, "nli")
nli_int8_preds = get_predictions(quantized_model_nli, dataset_rte_large, tokenizer_nli, "nli")
result_nli, table_nli = mcnemar_test(nli_fp32_preds, nli_int8_preds, nli_true)
print("\nNLI McNemar's test:", result_nli)
print("Contingency table:", table_nli)

Sentiment McNemar's test: pvalue      0.629058837890625
statistic   7.0
Contingency table: [[np.int64(446), np.int64(10)], [np.int64(7), np.int64(37)]]

NLI McNemar's test: pvalue      0.0592297037272794
statistic   3.5588235294117645
Contingency table: [[np.int64(157), np.int64(23)], [np.int64(11), np.int64(86)]]


In [29]:
from scipy import stats

def get_f1_scores(model, tokenizer, dataset):
    scores = []
    for example in dataset:
        predicted = get_answer(model, tokenizer, example['question'], example['context'])
        best_f1 = max(compute_f1(predicted, ans) for ans in example['answers']['text'])
        scores.append(best_f1)
    return scores

qa_fp32_scores = get_f1_scores(model_qa, tokenizer_qa, dataset_qa_large)
qa_int8_scores = get_f1_scores(quantized_model_qa, tokenizer_qa, dataset_qa_large)

t_stat, p_value = stats.ttest_rel(qa_fp32_scores, qa_int8_scores)
print(f"QA paired t-test: t = {t_stat:.4f}, p = {p_value:.6f}")

QA paired t-test: t = 4.6974, p = 0.000003


In [30]:
model_name_alb_sent = "textattack/albert-base-v2-SST-2"
tokenizer_alb_sent = AutoTokenizer.from_pretrained(model_name_alb_sent)
model_alb_sent = AutoModelForSequenceClassification.from_pretrained(model_name_alb_sent)

model_name_alb_nli = "anirudh21/albert-base-v2-finetuned-rte"
tokenizer_alb_nli = AutoTokenizer.from_pretrained(model_name_alb_nli)
model_alb_nli = AutoModelForSequenceClassification.from_pretrained(model_name_alb_nli)

model_name_alb_qa = "Firat/albert-base-v2-finetuned-squad"
tokenizer_alb_qa = AutoTokenizer.from_pretrained(model_name_alb_qa)
model_alb_qa = AutoModelForQuestionAnswering.from_pretrained(model_name_alb_qa)

print("All ALBERT models loaded!")
print("Sentiment labels:", model_alb_sent.config.id2label)
print("NLI labels:", model_alb_nli.config.id2label)

config.json:   0%|          | 0.00/732 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 46.7MB            

...

special_tokens_map.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 44.4MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

All ALBERT models loaded!
Sentiment labels: {0: 'LABEL_0', 1: 'LABEL_1'}
NLI labels: {0: 'LABEL_0', 1: 'LABEL_1'}


In [31]:
test_text = "This was an amazing experience!"
inputs = tokenizer_alb_sent(test_text, return_tensors="pt")
with torch.no_grad():
    outputs = model_alb_sent(**inputs)
print("Logits:", outputs.logits)
print("Predicted class:", torch.argmax(outputs.logits, dim=1).item())

Logits: tensor([[-4.1723,  3.6512]])
Predicted class: 1


In [32]:
quantized_model_alb_sent = torch.quantization.quantize_dynamic(model_alb_sent, {torch.nn.Linear}, dtype=torch.qint8)
quantized_model_alb_nli = torch.quantization.quantize_dynamic(model_alb_nli, {torch.nn.Linear}, dtype=torch.qint8)
quantized_model_alb_qa = torch.quantization.quantize_dynamic(model_alb_qa, {torch.nn.Linear}, dtype=torch.qint8)

print("All ALBERT models quantized!")

/tmp/ipykernel_502/1553311249.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model_alb_sent = torch.quantization.quantize_dynamic(model_alb_sent, {torch.nn.Linear}, dtype=torch.qint8)
/tmp/ipykernel_502/1553311249.py:2: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization

All ALBERT models quantized!


In [33]:
# Sentiment
fp32_acc_alb_sent = evaluate_model(model_alb_sent, dataset_large, tokenizer_alb_sent)
int8_acc_alb_sent = evaluate_model(quantized_model_alb_sent, dataset_large, tokenizer_alb_sent)
print(f"ALBERT Sentiment - FP32: {fp32_acc_alb_sent*100:.2f}% | INT8: {int8_acc_alb_sent*100:.2f}% | Drop: {(fp32_acc_alb_sent-int8_acc_alb_sent)*100:.2f} pts")

# NLI
fp32_acc_alb_nli = evaluate_nli_model(model_alb_nli, dataset_rte_large, tokenizer_alb_nli)
int8_acc_alb_nli = evaluate_nli_model(quantized_model_alb_nli, dataset_rte_large, tokenizer_alb_nli)
print(f"ALBERT NLI - FP32: {fp32_acc_alb_nli*100:.2f}% | INT8: {int8_acc_alb_nli*100:.2f}% | Drop: {(fp32_acc_alb_nli-int8_acc_alb_nli)*100:.2f} pts")

# QA
fp32_f1_alb_qa = evaluate_qa_model(model_alb_qa, tokenizer_alb_qa, dataset_qa_large)
int8_f1_alb_qa = evaluate_qa_model(quantized_model_alb_qa, tokenizer_alb_qa, dataset_qa_large)
print(f"ALBERT QA - FP32: {fp32_f1_alb_qa:.2f} | INT8: {int8_f1_alb_qa:.2f} | Drop: {(fp32_f1_alb_qa-int8_f1_alb_qa):.2f} pts")

ALBERT Sentiment - FP32: 93.00% | INT8: 47.00% | Drop: 46.00 pts
ALBERT NLI - FP32: 48.38% | INT8: 51.62% | Drop: -3.25 pts
ALBERT QA - FP32: 0.61 | INT8: 0.00 | Drop: 0.61 pts


In [34]:
test_text = "This was an amazing experience!"
inputs = tokenizer_alb_sent(test_text, return_tensors="pt")

with torch.no_grad():
    fp32_out = model_alb_sent(**inputs)
with torch.no_grad():
    int8_out = quantized_model_alb_sent(**inputs)

print("FP32 logits:", fp32_out.logits)
print("INT8 logits:", int8_out.logits)
print("FP32 prediction:", torch.argmax(fp32_out.logits, dim=1).item())
print("INT8 prediction:", torch.argmax(int8_out.logits, dim=1).item())

FP32 logits: tensor([[-4.1723,  3.6512]])
INT8 logits: tensor([[ 0.2944, -0.0754]])
FP32 prediction: 1
INT8 prediction: 0


In [35]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Fresh load, brand new objects, no reuse of anything from before
fresh_tokenizer = AutoTokenizer.from_pretrained("textattack/albert-base-v2-SST-2")
fresh_model = AutoModelForSequenceClassification.from_pretrained("textattack/albert-base-v2-SST-2")
fresh_model.eval()

fresh_quantized = torch.quantization.quantize_dynamic(fresh_model, {torch.nn.Linear}, dtype=torch.qint8)

test_text = "This was an amazing experience!"
inputs = fresh_tokenizer(test_text, return_tensors="pt")

with torch.no_grad():
    out_fp32 = fresh_model(**inputs)
    out_int8 = fresh_quantized(**inputs)

print("Fresh FP32 logits:", out_fp32.logits)
print("Fresh INT8 logits:", out_int8.logits)

Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

/tmp/ipykernel_502/3357567780.py:9: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  fresh_quantized = torch.quantization.quantize_dynamic(fresh_model, {torch.nn.Linear}, dtype=torch.qint8)


Fresh FP32 logits: tensor([[-4.1723,  3.6512]])
Fresh INT8 logits: tensor([[ 0.2944, -0.0754]])


In [36]:
fresh_tokenizer_nli = AutoTokenizer.from_pretrained("anirudh21/albert-base-v2-finetuned-rte")
fresh_model_nli = AutoModelForSequenceClassification.from_pretrained("anirudh21/albert-base-v2-finetuned-rte")
fresh_model_nli.eval()
fresh_quantized_nli = torch.quantization.quantize_dynamic(fresh_model_nli, {torch.nn.Linear}, dtype=torch.qint8)

example = dataset_rte_large[0]
inputs = fresh_tokenizer_nli(example['sentence1'], example['sentence2'], return_tensors="pt", truncation=True)

with torch.no_grad():
    out_fp32 = fresh_model_nli(**inputs)
    out_int8 = fresh_quantized_nli(**inputs)

print("Fresh NLI FP32 logits:", out_fp32.logits)
print("Fresh NLI INT8 logits:", out_int8.logits)

Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

/tmp/ipykernel_502/1909066780.py:4: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  fresh_quantized_nli = torch.quantization.quantize_dynamic(fresh_model_nli, {torch.nn.Linear}, dtype=torch.qint8)


Fresh NLI FP32 logits: tensor([[ 0.7426, -0.7975]])
Fresh NLI INT8 logits: tensor([[-0.4590, -0.3892]])


In [37]:
example = dataset_qa_large[0]
answer = get_answer(model_alb_qa, tokenizer_alb_qa, example['question'], example['context'])
print("Question:", example['question'])
print("Predicted:", answer)
print("Correct:", example['answers']['text'])

Question: Which NFL team represented the AFC at Super Bowl 50?
Predicted: denver broncos
Correct: ['Denver Broncos', 'Denver Broncos', 'Denver Broncos']


In [38]:
for i in range(5):
    example = dataset_qa_large[i]
    inputs = tokenizer_alb_qa(example['question'], example['context'], truncation=True, max_length=384)
    context_len = len(tokenizer_alb_qa(example['context'])['input_ids'])
    answer = get_answer(model_alb_qa, tokenizer_alb_qa, example['question'], example['context'])
    print(f"Example {i}: context token length={context_len}, predicted='{answer}', correct='{example['answers']['text'][0]}'")

Example 0: context token length=168, predicted='denver broncos', correct='Denver Broncos'
Example 1: context token length=168, predicted='denver broncos', correct='Carolina Panthers'
Example 2: context token length=168, predicted='santa clara, california', correct='Santa Clara, California'
Example 3: context token length=168, predicted='denver broncos', correct='Denver Broncos'
Example 4: context token length=168, predicted='gold', correct='gold'


In [39]:
low_score_examples = []

for i in range(50):
    example = dataset_qa_large[i]
    context_len = len(tokenizer_alb_qa(example['context'])['input_ids'])
    predicted = get_answer(model_alb_qa, tokenizer_alb_qa, example['question'], example['context'])
    best_f1 = max(compute_f1(predicted, ans) for ans in example['answers']['text'])

    if best_f1 < 0.1:
        low_score_examples.append((i, context_len, predicted, example['answers']['text'][0]))

print(f"Low-scoring examples: {len(low_score_examples)} out of 50")
for idx, clen, pred, correct in low_score_examples[:10]:
    print(f"idx={idx}, context_len={clen}, predicted='{pred}', correct='{correct}'")

Low-scoring examples: 21 out of 50
idx=1, context_len=168, predicted='denver broncos', correct='Carolina Panthers'
idx=5, context_len=168, predicted='', correct='"golden anniversary"'
idx=7, context_len=168, predicted='', correct='American Football Conference'
idx=8, context_len=168, predicted='', correct='"golden anniversary"'
idx=9, context_len=168, predicted='', correct='American Football Conference'
idx=13, context_len=168, predicted='', correct='Santa Clara'
idx=17, context_len=168, predicted='', correct='Santa Clara'
idx=19, context_len=168, predicted='', correct='24–10'
idx=23, context_len=168, predicted='denver broncos', correct='Carolina Panthers'
idx=26, context_len=168, predicted='', correct='Denver Broncos'


In [40]:
fp32_f1_check = evaluate_qa_model(model_alb_qa, tokenizer_alb_qa, dataset_qa_large.select(range(50)))
print(f"ALBERT FP32 F1 on first 50 examples: {fp32_f1_check*100:.2f}")

ALBERT FP32 F1 on first 50 examples: 54.06


In [41]:
subset = dataset_qa_large.select(range(50))

fp32_f1_50 = evaluate_qa_model(model_alb_qa, tokenizer_alb_qa, subset)
int8_f1_50 = evaluate_qa_model(quantized_model_alb_qa, tokenizer_alb_qa, subset)

print(f"FP32 F1 (n=50): {fp32_f1_50*100:.2f}")
print(f"INT8 F1 (n=50): {int8_f1_50*100:.2f}")
print(f"Drop: {(fp32_f1_50 - int8_f1_50)*100:.2f} pts")

FP32 F1 (n=50): 54.06
INT8 F1 (n=50): 0.06
Drop: 54.00 pts


In [42]:
# Isolated check: FP32 only, no quantization, small clean sample
subset_nli = dataset_rte_large.select(range(20))

for i in range(10):
    example = subset_nli[i]
    inputs = fresh_tokenizer_nli(example['sentence1'], example['sentence2'], return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = fresh_model_nli(**inputs)
    pred = torch.argmax(outputs.logits, dim=1).item()
    print(f"idx={i}, predicted={pred}, true={example['label']}, logits={outputs.logits.tolist()}")

idx=0, predicted=0, true=1, logits=[[0.7426495552062988, -0.797535240650177]]
idx=1, predicted=0, true=0, logits=[[0.7863157987594604, -0.7993735074996948]]
idx=2, predicted=0, true=1, logits=[[0.4022485613822937, -0.3250063359737396]]
idx=3, predicted=0, true=1, logits=[[0.26317542791366577, -0.20844237506389618]]
idx=4, predicted=0, true=0, logits=[[0.496924489736557, -0.48501458764076233]]
idx=5, predicted=0, true=0, logits=[[0.6790509223937988, -0.7199177145957947]]
idx=6, predicted=0, true=0, logits=[[0.5582742094993591, -0.5500783324241638]]
idx=7, predicted=0, true=0, logits=[[0.6981586813926697, -0.6951555609703064]]
idx=8, predicted=0, true=0, logits=[[0.7119476795196533, -0.7516555190086365]]
idx=9, predicted=0, true=0, logits=[[0.34932592511177063, -0.29952913522720337]]


In [43]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name_alb_nli2 = "Alireza1044/albert-base-v2-rte"
tokenizer_alb_nli2 = AutoTokenizer.from_pretrained(model_name_alb_nli2)
model_alb_nli2 = AutoModelForSequenceClassification.from_pretrained(model_name_alb_nli2)
model_alb_nli2.eval()

print("Loaded. Label map:", model_alb_nli2.config.id2label)


config.json:   0%|          | 0.00/916 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 46.8MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

Loaded. Label map: {0: 'LABEL_0', 1: 'LABEL_1'}
